In [1]:
"""
Simple EU (Ireland) driving licence verifier using PaddleOCR.

Requirements:
    pip install paddlepaddle paddleocr Pillow

Directory layout for testing:
    sample_0.png
    sample_1.png
"""

# ---------- imports ----------

from paddleocr import PaddleOCR        # OCR engine
from PIL import Image                  # to get image size if needed
import re                              # regex for pattern checks
from typing import List, Dict, Any     # type hints
import os
import random
from datetime import datetime
from paddleocr import PaddleOCR
import cv2
import logging

Checking connectivity to the model hosters, this may take a while. To bypass this check, set `DISABLE_MODEL_SOURCE_CHECK` to `True`.


In [2]:


# Set up logging to see what's happening
logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

class DrivingLicenseVerifier:
    def __init__(self, lang='en'):
        # Initialize PaddleOCR
        # use_angle_cls=True helps if the image is slightly rotated
        self.ocr = PaddleOCR(use_angle_cls=True, lang=lang)
        
    def extract_text(self, image_path):
        """
        Runs PaddleOCR on the image and returns a list of detected strings.
        """
        logging.info(f"Processing image: {image_path}")
        result = self.ocr.ocr(image_path)
        
        texts = result[0]["rec_texts"]
                
        return texts

    def parse_data(self, texts):
        """
        Simplest possible parser:
        - Join OCR lines into one string
        - Check presence of markers: 1,2,3,4a,4b,4c,4d,5,7,8,9
        - Return a dict of marker -> True/False
        """
        # Join all text and normalize spaces
        text = " ".join(texts or [])
        text = re.sub(r"\s+", " ", text).strip().lower()
    
        markers = ["1", "2", "3", "4a", "4b", "4c", "4d", "5", "7", "8", "9"]
        found = {}
    
        for m in markers:
            if m.isdigit():
                # Match like: "1." "1," "1:" "1-" "1)" etc. (anywhere)
                pattern = rf"(?<!\d){m}\s*[\.\):,\-]"
            else:
                # Match like: "4a" "4a." "4a," "4a:" etc. (anywhere)
                pattern = rf"\b{re.escape(m)}\b\s*[\.\):,\-]?"
    
            found[m] = re.search(pattern, text, flags=re.IGNORECASE) is not None
    
        return {"markers": found}


    def verify_data(self, parsed_data):
        """
        Simplest possible verification:
        - Valid if ALL required markers are present
        - Report missing markers
        """
        required = ["1", "2", "3", "4a", "4b", "4c", "4d", "5", "7", "8", "9"]
        markers = (parsed_data or {}).get("markers", {})
    
        missing = [m for m in required if not markers.get(m, False)]
        found = [m for m in required if markers.get(m, False)]
    
        return {
            "is_valid_format": len(missing) == 0,
            "found_markers": found,
            "missing_markers": missing,
        }



In [3]:
# --- Execution Example ---

def process_license(image_path):
    parser = DrivingLicenseVerifier()
    
    # 1. Extract
    raw_text = parser.extract_text(image_path)
    print(f"\n--- Raw OCR Text for {image_path} ---\n{raw_text}")
    
    # 2. Parse
    structured_data = parser.parse_data(raw_text)
    print(f"\n--- Parsed Data ---\n{structured_data}")
    
    # 3. Verify
    verification_report = parser.verify_data(structured_data)
    print(f"\n--- Verification Report ---\n{verification_report}")
    
    return structured_data, verification_report

# To run this code, simply call the function with your image file path:
# process_license("generated_license_0.jpg")
# process_license("generated_license_1.jpg")
# process_license("generated_license_2.jpg")

In [4]:
def _collect_images(root):
    exts = {'.jpg', '.jpeg', '.png', '.bmp', '.gif', '.tiff'}
    imgs = []
    for dirpath, _, filenames in os.walk(root):
        for fn in filenames:
            if os.path.splitext(fn)[1].lower() in exts:
                imgs.append(os.path.join(dirpath, fn))
    return imgs

def sample_pos_neg(n, base_dir='.', pos_sub='data/Original', neg_sub='data/random_doc_images', seed=None):
    """
    Return two lists: (positive_samples, negative_samples).
    - n: number of samples per class (if fewer available, returns all).
    - base_dir: base path containing the data folders (default: current dir).
    - seed: optional int for deterministic sampling.
    """
    if seed is not None:
        random.seed(seed)

    pos_root = os.path.join(base_dir, pos_sub)
    neg_root = os.path.join(base_dir, neg_sub)

    pos_imgs = _collect_images(pos_root)
    neg_imgs = _collect_images(neg_root)

    if not pos_imgs:
        raise FileNotFoundError(f"No positive images found in {pos_root}")
    if not neg_imgs:
        raise FileNotFoundError(f"No negative images found in {neg_root}")

    k_pos = min(n, len(pos_imgs))
    k_neg = min(n, len(neg_imgs))

    pos_sample = random.sample(pos_imgs, k_pos)
    neg_sample = random.sample(neg_imgs, k_neg)

    return pos_sample, neg_sample


In [5]:
pos, neg = sample_pos_neg(10, base_dir='.', seed=42)

sample_paths = pos + neg

for path in sample_paths:
    print("=" * 80)
    print(f"Verifying image: {path}")
    result = process_license(path)

Verifying image: .\data/Original\generated_license_656.png


C:\Users\rahma\AppData\Local\Temp\ipykernel_32120\1316587943.py:8: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  self.ocr = PaddleOCR(use_angle_cls=True, lang=lang)
C:\Users\rahma\anaconda3\envs\document_verification\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rahma\.paddlex\official_models\PP-LCNet_x1_0_doc_ori`.
Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rahma\.paddlex\offi


--- Raw OCR Text for .\data/Original\generated_license_656.png ---
['CEADUNAS TIOMANA', 'DRIVING LICENCE', 'ÉIRE', 'IRL', 'Lcagan', '0]π', 'Aontais Eorpoigh', 'Europcon', 'Union', 'IRELAND', 'nodel', 'Cmenooncrno aa ympoooono', '1.', 'Hale', 'NCC', 'ivieova otnc Pemloo de condoccioo', '2.', 'Pamela', 'booduccion fbdidohg prohaz flaralcoce', '3.', '26.06.43', 'sEdcae Korehort Pohcerochetn Juhioba', '$AScn pönonna.Uoimig de gondoice', '4a.', '13.01.75', '4c. NATIONAL TRANSPORTIAUTHORITYpliocilon', '4b.', '24.04.13', '4d. 821575970opliociha Valiuoinjo paiyonojiovos', '5.', 'GPFXH3LSCO', 'Vezelbi ongerioly Lloeroje tae-sewgon', 'iela f6s-aoangen fajbonots fcoom joody', '7.', 'wholjs Prouo janly Canla de condocan', 'dDe', 'condugio', 'Hennis do', 'condococe', 'dhecerai', 'Vidiloky', 'pouchen', 'ao1ko', '8.', '873 Thomas Glens', 'Vocolero', 'cooulionjo', 'A', 'CO', 'Pennsylvania Kyrgyz', 'z Republicugr661', 'sce', 'MACTCNCIOO', '0O', '90J50', 'hoos', 'te', 'conchoociors', 'AUdK', 'RGiow', '

Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rahma\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rahma\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rahma\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rahma\.paddlex\official_models\en_PP-OCRv5_mobile_rec`.
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rahma\.paddlex


--- Raw OCR Text for .\data/Original\generated_license_1408.png ---
['CEADUNAS TIOMANA', 'DRIVING LICENCE', 'ÉIRE', 'IRL', 'Leagan', 'an Aontois Eorpjigh', 'Europeon', 'Unlon oodel', 'IRELAND', 'Cmenojoncrao aa ymponooono', '1.', 'Shaw', 'NOC', 'iveova otnc', 'Pemioo', 'do', 'condoccioo', '2.', 'Kevin', 'cooduccion fbdidohg prohax', 'fiaralcoce', '25.07.68', 'sdcer Korehort Pohcerochetn Juhdoba', '3.', 'A8gnI Döouung Uoimie de', '4a, 08.07.29', '4c. DANISHROAD DIRECTQRATEltale apliocilon', 'condoica', '4b,31.10.93', '4d. 234203357opliociha Valiuoinjo paiynojiovns', '5. SZG7FKHNP', 'Vezelbi ongoroly Llceroje tBe-sewgon', 'Iela f0s-aceugen fujboots fcoom foody', '7.', 'holjs Prouo janly Canla de condocan', 'de condugio Hennis', 'do', 'condococe', 'decera Viidiloky', 'pouacHon', '401ko', '8.953 Adam Rapid', 'Yocoleko dooulionjo', 'AJ', 'CO', 'Indiana Cook Islands 229570ti', 'Hirlcora', 'SCe', 'INCTONCIOO', '0O', '9DJ150', 'hoos', 'de', 'conchoocio18', 'AJdK', 'RGiow', '9. BE/C/B/A/A2', '

...


--- Raw OCR Text for .\data/random_doc_images\book_pages\25.jpg ---
['138', 'MONETBRLL', "CIRMBI'S HOLE", '190', 'haps having decided that Tejada is beginning to worry abour the', "I think he's joking hut he's not Mecir was born with twe", 'one-way trip to Mexico, tries to come back to the same plaz', "clubtet. As a child he'd had operatsons to correct them hut be", "which he really shouldn't do. Teiada mects the pitch with s gog", 'soill walked with a limp. Somehow he had rurned his detormiry', 'crude stroke and crushes it into the left field bleachers. Yankseg', "ino an advantagn. His strange delivery-he wasn't ahle to push off", 'Oakland 3. Goliath, meer David.', 'the mound with his right foot-put an unusually violent spin on', 'Two innings later, in the bottom of the sixth, Daend Jue', 'hs seeewball The pitch had proven to be ruthlessly eifectrve', 'leads off the inning again, and this cime cdraws a walk from Wels', 'sgatnst lett-handed hitters.', 'Minutes later he crosscs-the pla

Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rahma\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rahma\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rahma\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rahma\.paddlex\official_models\en_PP-OCRv5_mobile_rec`.
Creating model: ('PP-LCNet_x1_0_doc_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rahma\.paddlex


--- Raw OCR Text for .\data/random_doc_images\national_certificates\24.jpeg ---
['', 'CERTIFICATE OF ACCEPTANCE', 'aidl. 20r1ld Tharphae Chera']

--- Parsed Data ---
{'markers': {'1': False, '2': False, '3': False, '4a': False, '4b': False, '4c': False, '4d': False, '5': False, '7': False, '8': False, '9': False}}

--- Verification Report ---
{'is_valid_format': False, 'found_markers': [], 'missing_markers': ['1', '2', '3', '4a', '4b', '4c', '4d', '5', '7', '8', '9']}
Verifying image: .\data/random_doc_images\newspapers\36.jpg


Creating model: ('UVDoc', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rahma\.paddlex\official_models\UVDoc`.
Creating model: ('PP-LCNet_x1_0_textline_ori', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rahma\.paddlex\official_models\PP-LCNet_x1_0_textline_ori`.
Creating model: ('PP-OCRv5_server_det', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rahma\.paddlex\official_models\PP-OCRv5_server_det`.
Creating model: ('en_PP-OCRv5_mobile_rec', None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\rahma\.paddlex\official_models\en_PP-OCRv5_mobile_rec`.



--- Raw OCR Text for .\data/random_doc_images\newspapers\36.jpg ---
['STAREDU, SUNDAY 4 OCTOBER 2020', 'Opinion 7', 'Live&Learn', '1 Wing Lam', 'WE are entering a digital era.', 'Digitally skilled', "The world's most valuable companies", 'ate compelling digital experiences.', 'are no longer banks or energy', 'Graduates that are able to apply a', 'companies, but technology companies.', 'design thinking approach will be highly', 'Accenture chief executive officer (CEO)', 'sought after by organisations undergoing', 'Pierre Nanterme noted that "digital is the', 'digital transformation.', 'main reason just over half of the', 'companies in the Fortune S00 have', 'The workforce of tomorrow needs', '> Developing code', 'disappeared since the year 2000.', 'With so much emphasis on digital,', 'talents who are knowledgeable in all', 'Coding is the second most important', "it's clear that today's graduates not only", 'language you can-learn," said Apple CEO', 'need the hard and soft skills that',